# Step 2 — Peristimulus visualization (single-trial)

**Input contract** (produced by Step 1 Block 6, see `step1_phantom_video_analysis.ipynb`):
- `<stem>_combined.csv` on a uniform 20 kHz grid in absolute NIDAQ seconds
- required columns: `time_s`, `rescaled_intensity`, `wingbeat_amplitude`, `wingbeat_frequency`, `inferred_flight_power`
- matching `<stem>.mat` where `Data[0,:]` = NIDAQ absolute time, `Data[6,:]` = optogenetic TTL

**What this notebook does**
1. Batch-finds every `*_combined.csv` under `DATA_DIR` and pairs it with `<stem>.mat`.
2. Reads `Data[6,:]` as opto TTL; robustly extracts the main pulse train.
3. Sets `onset_s` = first rise of the main train, `offset_s` = last fall.
4. Trims `<stem>_combined.csv` to `[onset_s - 2 s, offset_s + 5 s]`.
5. Plots 5 stacked panels (relative time, 0 = onset) and saves:
   - `<stem>_peristimulus_window.csv`
   - `<stem>_peristimulus_plot.svg`
   - `<stem>_peristimulus_plot.png`

**Opto stimulus assumed** (drives detection tuning): 200 Hz, 3 ms pulses, 3 s or 10 s total train.

---

## 中文说明

**用途**：这是分析流程的第二步（单次实验 / single-trial 版本）。它围绕光遗传（optogenetic）刺激的起止时刻，把每一次实验（trial）的行为信号截取成一个"刺激前后窗口"（peristimulus window），并画成 5 联图，便于逐次实验查看刺激引起的反应。

**预期输入**：
- 文件类型：`<stem>_combined.csv`（由 Step 1 的 Block 6 生成，已重采样到均匀的 20 kHz 时间网格，时间轴是 NIDAQ 绝对秒）；以及与之同名的 `<stem>.mat`。两者通过同一个基础文件名（base name）`<stem>` 配对。
- 这里只需关注基础名之后的后缀约定：行为数据用 `_combined.csv` 后缀，原始采集用同名 `.mat`。
- CSV 必需列：`time_s`（时间，秒）、`rescaled_intensity`（气门开度 / spiracle aperture，0–1 归一化）、`wingbeat_amplitude`（翅拍幅度，rad）、`wingbeat_frequency`（翅拍频率，Hz）、`inferred_flight_power`（推算飞行功率，W/kg，由翅拍幅度与频率推算，并非二者简单相乘）。
- `.mat` 中 `Data[0,:]` = NIDAQ 绝对时间，`Data[6,:]` = 光遗传刺激的 TTL 方波信号。

**预期输出**（与每个输入同目录、同基础名）：
- `<stem>_peristimulus_window.csv`：截取后的窗口数据（含绝对时间、相对刺激起点的时间、opto_on 标记及 4 个行为信号）。
- `<stem>_peristimulus_plot.svg` 和 `.png`：5 联图（刺激条 + 4 个行为信号，横轴为相对刺激起点的时间，0 = 刺激起点）。

**对刺激的假设**（用于调参做脉冲检测）：200 Hz、3 ms 脉宽、总时长 3 s 或 10 s 的脉冲串。

### Block 1 — Imports & parameters

Loads libraries and sets every knob used downstream: `DATA_DIR` (folder to scan), the opto-TTL detection thresholds (tuned for 200 Hz / 3 ms pulses / 3–10 s trains), and the peristimulus window padding (`PRE_S`, `POST_S`). Edit `DATA_DIR` to point at the folder of Step 1 outputs you want to process.

**中文**：导入所需库，并设置后续所有可调参数：`DATA_DIR`（要扫描的文件夹）、光遗传 TTL 脉冲的检测阈值（针对 200 Hz / 3 ms 脉宽 / 3–10 s 脉冲串调好）、以及刺激前后窗口的留白时长（`PRE_S`、`POST_S`）。运行新数据时只需把 `DATA_DIR` 改成对应的 Step 1 输出文件夹。

In [15]:
# Block 1 — imports + parameters
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt

# Folder to scan for Step 1 outputs
DATA_DIR  = Path(r"C:\Users\Lylah\Desktop\data_processing\SpINB_ChR\SpINB_ChR_3s")
RECURSIVE = False

# Opto TTL detection — tuned for 200 Hz / 3 ms pulses / 3–10 s train
OPTO_TTL_ROW     = 6
THRESHOLD        = 1.5     # V (same as Step 1 Blocks 2 and 5)
DEADTIME_S       = 0.0005   # 0.5 ms debounce (< 5 ms inter-pulse period at 200 Hz)
MIN_PULSE_WIDTH  = 0.0005  # 0.5 ms  (real pulses ~3 ms)
MAX_PULSE_WIDTH  = 0.006   # 6 ms    (real pulses ~3 ms)
MAX_GAP_S        = 0.020   # 20 ms   (in-train gap ~2 ms; inter-train gaps are seconds)
MIN_TRAIN_PULSES = 100     # 3 s train -> 600 pulses; 10 s -> 2000

# Peristimulus window around detected onset/offset
PRE_S  = 5.0
POST_S = 5.0

### Block 2 — Input discovery & loaders

Two helpers. `find_step2_jobs` scans `DATA_DIR` for every `*_combined.csv` and pairs each with its same-named `.mat` (warning and skipping any CSV whose `.mat` is missing). `load_step2_inputs` reads the CSV (checking the required columns are present) and pulls the absolute time vector (`Data[0,:]`) and the opto TTL trace (`Data[6,:]`) out of the `.mat`.

**中文**：两个辅助函数。`find_step2_jobs` 扫描 `DATA_DIR` 下的每个 `*_combined.csv`，并与同名 `.mat` 配对（若缺少对应 `.mat` 则告警并跳过）。`load_step2_inputs` 读取 CSV（并检查必需列是否齐全），同时从 `.mat` 中取出绝对时间向量（`Data[0,:]`）和光遗传 TTL 信号（`Data[6,:]`）。

In [9]:
# Block 2 — input discovery + loaders

REQUIRED_COLS = [
    "time_s",
    "rescaled_intensity",
    "wingbeat_amplitude",
    "wingbeat_frequency",
    "inferred_flight_power",
]


def find_step2_jobs(data_dir, recursive=False):
    """Find every <stem>_combined.csv under data_dir and pair it with <stem>.mat.

    Returns a list of (mat_file, combined_csv) as absolute Path pairs.
    Warns and skips any CSV whose matching MAT file is missing.
    """
    data_dir = Path(data_dir)
    it = data_dir.rglob("*_combined.csv") if recursive else data_dir.glob("*_combined.csv")
    jobs = []
    for csv_path in sorted(it):
        name = csv_path.name
        if not name.endswith("_combined.csv"):
            continue
        stem = name[: -len("_combined.csv")]
        mat_path = csv_path.with_name(stem + ".mat")
        if not mat_path.exists():
            print(f"[warn] no matching MAT for {csv_path.name} (expected {mat_path.name}) — skipping")
            continue
        jobs.append((mat_path, csv_path))
    return jobs


def load_step2_inputs(mat_file, combined_csv):
    """Load the combined CSV and the MAT file; return (df, t_abs, opto_ttl)."""
    combined_df = pd.read_csv(combined_csv)
    missing = [c for c in REQUIRED_COLS if c not in combined_df.columns]
    if missing:
        raise ValueError(f"{combined_csv} missing required columns: {missing}")

    mat = scipy.io.loadmat(str(mat_file), squeeze_me=True, struct_as_record=False)
    Data = mat["Data"]
    if Data.ndim != 2 or Data.shape[0] <= OPTO_TTL_ROW:
        raise ValueError(
            f"Data shape {Data.shape} does not contain row {OPTO_TTL_ROW} (opto TTL)"
        )
    t_abs = np.asarray(Data[0, :], dtype=float)
    opto_ttl = np.asarray(Data[OPTO_TTL_ROW, :], dtype=float)
    return combined_df, t_abs, opto_ttl

### Block 3 — Detect the main opto pulse train

`detect_opto_train` finds the start and end of the real stimulus train from the noisy TTL trace. It thresholds the signal, finds rising/falling edges, debounces them, pairs each rise with its fall and rejects pulses whose width is outside the expected 0.5–6 ms range, groups surviving pulses into trains (a gap > 20 ms starts a new train), drops trains with too few pulses, and picks the longest-duration train as the real stimulus. Returns `onset_s`, `offset_s`, duration and pulse count.

**中文**：`detect_opto_train` 从带噪的 TTL 信号中找出真正刺激脉冲串的起止时刻。流程：阈值二值化 → 找上升/下降沿 → 去抖动 → 把每个上升沿与其后的下降沿配对，并剔除脉宽不在预期 0.5–6 ms 区间内的脉冲 → 把相邻脉冲按间隔分组成"脉冲串"（间隔 > 20 ms 即视为新串）→ 丢弃脉冲数过少的串 → 选时长最长的那一串作为真正刺激。返回刺激起点 `onset_s`、终点 `offset_s`、时长和脉冲数。

In [10]:
# Block 3 — detect the main opto pulse train

def detect_opto_train(
    t_abs,
    ttl,
    threshold=THRESHOLD,
    deadtime_s=DEADTIME_S,
    min_pulse_width_s=MIN_PULSE_WIDTH,
    max_pulse_width_s=MAX_PULSE_WIDTH,
    max_gap_s=MAX_GAP_S,
    min_train_pulses=MIN_TRAIN_PULSES,
):
    """Find onset/offset of the main optogenetic pulse train.

    Steps:
      1. threshold
      2. find rises and falls (handle starts-high / ends-high)
      3. debounce rises
      4. pair each rise with the next fall, reject widths outside [min, max]
      5. group consecutive pulses into trains by low gap <= max_gap_s
      6. drop trains with < min_train_pulses
      7. main train = longest duration (tiebreak: most pulses)
    """
    t_abs = np.asarray(t_abs, dtype=float)
    ttl = np.asarray(ttl, dtype=float)
    if t_abs.shape != ttl.shape or t_abs.size < 2:
        raise ValueError("t_abs and ttl must have matching shape and >= 2 samples")

    # 1. threshold
    binary = (ttl >= threshold).astype(np.int8)

    # 2. edges
    d = np.diff(binary)
    rises = np.where(d == 1)[0] + 1   # first HIGH sample
    falls = np.where(d == -1)[0] + 1  # first LOW sample after the pulse
    if binary[0] == 1:
        print("[warn] opto TTL starts HIGH at first sample — ignoring any partial leading pulse")
    if binary[-1] == 1:
        print("[warn] opto TTL ends HIGH at last sample — appending synthetic fall at end of recording")
        falls = np.append(falls, len(binary) - 1)
    n_raw_rises = len(rises)

    # 3. debounce rises by deadtime_s
    if n_raw_rises > 0:
        rise_times_raw = t_abs[rises]
        keep_mask = np.ones(n_raw_rises, dtype=bool)
        last_kept_time = rise_times_raw[0]
        for i in range(1, n_raw_rises):
            if rise_times_raw[i] - last_kept_time < deadtime_s:
                keep_mask[i] = False
            else:
                last_kept_time = rise_times_raw[i]
        rises = rises[keep_mask]
    n_debounced_rises = len(rises)

    # 4. pair each rise with the next fall; filter on pulse width
    pulses = []  # list of (rise_time, fall_time)
    fi = 0
    for r in rises:
        while fi < len(falls) and falls[fi] <= r:
            fi += 1
        if fi >= len(falls):
            break
        f = falls[fi]
        pulses.append((t_abs[r], t_abs[f]))
        fi += 1
    n_paired = len(pulses)

    good_pulses = [
        (rt, ft) for (rt, ft) in pulses
        if min_pulse_width_s <= (ft - rt) <= max_pulse_width_s
    ]
    n_good = len(good_pulses)

    # 5. group pulses into trains
    trains = []
    if n_good > 0:
        current = [good_pulses[0]]
        for prev, nxt in zip(good_pulses[:-1], good_pulses[1:]):
            low_gap = nxt[0] - prev[1]
            if low_gap <= max_gap_s:
                current.append(nxt)
            else:
                trains.append(current)
                current = [nxt]
        trains.append(current)
    n_trains_all = len(trains)

    # 6. drop trains shorter than min_train_pulses
    trains_big = [tr for tr in trains if len(tr) >= min_train_pulses]
    n_trains_big = len(trains_big)

    if n_trains_big == 0:
        raise RuntimeError(
            "No valid opto pulse train found. "
            f"Counts: raw_rises={n_raw_rises}, post_debounce={n_debounced_rises}, "
            f"paired={n_paired}, width_pass={n_good}, trains={n_trains_all}, "
            f"trains_with>={min_train_pulses}_pulses={n_trains_big}."
        )

    # 7. main train = longest duration, tiebreak by pulse count
    def train_key(tr):
        return (tr[-1][1] - tr[0][0], len(tr))

    main = max(trains_big, key=train_key)
    onset_s = float(main[0][0])
    offset_s = float(main[-1][1])
    duration_s = offset_s - onset_s

    return {
        "onset_s": onset_s,
        "offset_s": offset_s,
        "duration_s": duration_s,
        "n_pulses": len(main),
        "n_trains": n_trains_big,
    }

### Block 4 — Trim to the peristimulus window

`trim_peristimulus_window` keeps only the rows of the combined dataframe whose `time_s` falls in `[onset − PRE_S, offset + POST_S]` (clipping to the recording bounds with a warning if the window runs off either end). It adds three convenience columns: `time_s_abs` (the original absolute time), `time_s_rel_onset` (time relative to stimulus onset, so 0 = onset), and `opto_on` (boolean flag for samples inside the stimulus).

**中文**：`trim_peristimulus_window` 只保留合并数据中 `time_s` 落在 `[onset − PRE_S, offset + POST_S]` 区间内的行（若窗口超出记录范围则告警并裁剪到记录边界）。它新增三列：`time_s_abs`（原始绝对时间）、`time_s_rel_onset`（相对刺激起点的时间，0 即刺激起点）、`opto_on`（标记该采样点是否处于刺激期间的布尔值）。

In [11]:
# Block 4 — trim combined df to the peristimulus window

def trim_peristimulus_window(df, onset_s, offset_s, pre_s=PRE_S, post_s=POST_S):
    """Mask df by time_s to [onset - pre_s, offset + post_s] and add relative-time columns."""
    t_min = float(df["time_s"].iloc[0])
    t_max = float(df["time_s"].iloc[-1])
    plot_t0 = onset_s - pre_s
    plot_t1 = offset_s + post_s
    if plot_t0 < t_min:
        print(f"[warn] requested window starts at {plot_t0:.3f}s but recording starts at {t_min:.3f}s — clipping")
        plot_t0 = t_min
    if plot_t1 > t_max:
        print(f"[warn] requested window ends at {plot_t1:.3f}s but recording ends at {t_max:.3f}s — clipping")
        plot_t1 = t_max

    mask = (df["time_s"] >= plot_t0) & (df["time_s"] <= plot_t1)
    out = df.loc[mask].copy()
    out["time_s_abs"] = out["time_s"]
    out["time_s_rel_onset"] = out["time_s"] - onset_s
    out["opto_on"] = (out["time_s"] >= onset_s) & (out["time_s"] <= offset_s)
    return out, plot_t0, plot_t1

### Block 5 — Five-panel peristimulus figure

`plot_peristimulus` draws a stacked figure aligned to stimulus onset (x = 0): a thin top strip shading the opto-on window, then four behavioral signals (spiracle aperture, wingbeat amplitude, wingbeat frequency, inferred flight power). Dashed vertical lines mark onset and offset, and a faint red band marks the stimulus window on every panel. The figure is saved as both SVG and PNG.

**中文**：`plot_peristimulus` 画出以刺激起点对齐（x = 0）的纵向多联图：顶部一条窄带用阴影标出刺激开启的时间段，下面是四个行为信号（气门开度、翅拍幅度、翅拍频率、推算飞行功率）。虚线竖线标出刺激起止时刻，每个子图都用淡红色带标出刺激窗口。图同时保存为 SVG 与 PNG。

In [12]:
# Block 5 — 5-panel peristimulus figure

def plot_peristimulus(df_trim, onset_s, offset_s, out_svg, out_png, title=None,
                     pre_s=PRE_S, post_s=POST_S):
    duration_s = offset_s - onset_s

    fig = plt.figure(figsize=(14, 13))
    gs = fig.add_gridspec(5, 1, height_ratios=[0.4, 1.0, 1.0, 1.0, 1.0], hspace=0.12)
    ax_opto = fig.add_subplot(gs[0, 0])
    ax_intensity = fig.add_subplot(gs[1, 0], sharex=ax_opto)
    ax_amp       = fig.add_subplot(gs[2, 0], sharex=ax_opto)
    ax_freq      = fig.add_subplot(gs[3, 0], sharex=ax_opto)
    ax_power     = fig.add_subplot(gs[4, 0], sharex=ax_opto)
    data_axes = [ax_intensity, ax_amp, ax_freq, ax_power]

    # Panel 1 — opto-on shaded strip
    ax_opto.axvspan(0.0, duration_s, color="red", alpha=0.35)
    ax_opto.set_yticks([])
    ax_opto.set_ylabel("opto")
    ax_opto.set_ylim(0, 1)
    plt.setp(ax_opto.get_xticklabels(), visible=False)

    # Panels 2–5 — behavioral signals
    x = df_trim["time_s_rel_onset"].values
    panels = [
        (ax_intensity, "rescaled_intensity",    "Spiracle aperture",    ""),
        (ax_amp,       "wingbeat_amplitude",    "Wingbeat amplitude",  "rad"),
        (ax_freq,      "wingbeat_frequency",    "Wingbeat frequency",  "Hz"),
        (ax_power,     "inferred_flight_power", "Inferred flight power", "W/kg"),
    ]
    for ax, col, label, unit in panels:
        y = df_trim[col].values
        ax.axvspan(0.0, duration_s, color="red", alpha=0.10, zorder=0)
        ax.plot(x, y, lw=0.9, color="#1f77b4")
        ax.axvline(0.0, color="k", linestyle="--", alpha=0.6, lw=0.8)
        ax.axvline(duration_s, color="k", linestyle="--", alpha=0.6, lw=0.8)
        ax.grid(True, linestyle="--", alpha=0.35)
        ylabel = f"{label} ({unit})" if unit else label
        ax.set_ylabel(ylabel)

    # Dashed markers on the opto strip too
    ax_opto.axvline(0.0, color="k", linestyle="--", alpha=0.6, lw=0.8)
    ax_opto.axvline(duration_s, color="k", linestyle="--", alpha=0.6, lw=0.8)

    # Hide x-tick labels on all but bottom panel
    for ax in [ax_intensity, ax_amp, ax_freq]:
        plt.setp(ax.get_xticklabels(), visible=False)

    ax_power.set_xlabel("Time relative to opto onset (s)")
    ax_power.set_xlim(-pre_s, duration_s + post_s)

    if title:
        fig.suptitle(title, fontsize=10)
        fig.subplots_adjust(top=0.96)

    fig.savefig(out_svg, format="svg", bbox_inches="tight")
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)

### Block 6 — Save the trimmed CSV

`save_step2_outputs` writes the trimmed window to `<stem>_peristimulus_window.csv` using a fixed column schema (`time_s_abs`, `time_s_rel_onset`, `opto_on`, plus the four behavioral signals) and prints the detected onset/offset/duration and row count for a quick sanity check.

**中文**：`save_step2_outputs` 按固定的列结构（`time_s_abs`、`time_s_rel_onset`、`opto_on` 及四个行为信号）把截取后的窗口写入 `<stem>_peristimulus_window.csv`，并打印检测到的起点/终点/时长和行数，便于快速核对。

In [13]:
# Block 6 — save trimmed CSV with the design-doc schema

OUTPUT_COLS = [
    "time_s_abs",
    "time_s_rel_onset",
    "opto_on",
    "rescaled_intensity",
    "wingbeat_amplitude",
    "wingbeat_frequency",
    "inferred_flight_power",
]


def save_step2_outputs(df_trim, meta, out_csv, plot_t0, plot_t1):
    out = df_trim[OUTPUT_COLS].copy()
    out.to_csv(out_csv, index=False)
    print(
        f"  onset_s={meta['onset_s']:.4f}  offset_s={meta['offset_s']:.4f}  "
        f"duration_s={meta['duration_s']:.4f}  n_pulses={meta['n_pulses']}\n"
        f"  plot_t0={plot_t0:.4f}  plot_t1={plot_t1:.4f}  rows={len(out)}\n"
        f"  saved: {out_csv}"
    )

### Block 7 — Batch runner

The driver. It lists all jobs in `DATA_DIR`, then for each one runs the full pipeline — load inputs → detect the opto train → trim the window → plot → save CSV/SVG/PNG — printing per-trial results. Any trial that errors is caught and reported with `[SKIP]` so one bad file does not halt the batch.

**中文**：主控单元。它列出 `DATA_DIR` 中的所有任务，对每个任务依次执行完整流程——加载输入 → 检测光遗传脉冲串 → 截取窗口 → 绘图 → 保存 CSV/SVG/PNG——并打印每次实验的结果。若某次实验出错，会被捕获并以 `[SKIP]` 标出，从而避免单个坏文件中断整个批处理。

In [16]:
# Block 7 — batch runner

jobs = find_step2_jobs(DATA_DIR, recursive=RECURSIVE)
print(f"Found {len(jobs)} combined CSV(s) in {DATA_DIR}")

for mat_file, combined_csv in jobs:
    stem_name = Path(mat_file).stem
    print(f"\n=== {stem_name} ===")
    try:
        combined_df, t_abs, opto_ttl = load_step2_inputs(mat_file, combined_csv)
        meta = detect_opto_train(t_abs, opto_ttl)
        df_trim, plot_t0, plot_t1 = trim_peristimulus_window(
            combined_df, meta["onset_s"], meta["offset_s"], PRE_S, POST_S
        )

        stem_path = os.path.splitext(str(mat_file))[0]
        out_csv = stem_path + "_peristimulus_window.csv"
        out_svg = stem_path + "_peristimulus_plot.svg"
        out_png = stem_path + "_peristimulus_plot.png"

        plot_peristimulus(
            df_trim,
            meta["onset_s"],
            meta["offset_s"],
            out_svg=out_svg,
            out_png=out_png,
            title=stem_name,
        )
        save_step2_outputs(df_trim, meta, out_csv, plot_t0, plot_t1)
        print(f"  saved: {out_svg}")
        print(f"  saved: {out_png}")
    except Exception as e:
        print(f"  [SKIP] {type(e).__name__}: {e}")

Found 8 combined CSV(s) in C:\Users\Lylah\Desktop\data_processing\SpINB_ChR\SpINB_ChR_3s

=== 2026_0507_095718_SpINB_ChR_IS46338_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON ===
  onset_s=30.0000  offset_s=33.0000  duration_s=3.0000  n_pulses=968
  plot_t0=25.0000  plot_t1=38.0000  rows=260001
  saved: C:\Users\Lylah\Desktop\data_processing\SpINB_ChR\SpINB_ChR_3s\2026_0507_095718_SpINB_ChR_IS46338_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  saved: C:\Users\Lylah\Desktop\data_processing\SpINB_ChR\SpINB_ChR_3s\2026_0507_095718_SpINB_ChR_IS46338_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_peristimulus_plot.svg
  saved: C:\Users\Lylah\Desktop\data_processing\SpINB_ChR\SpINB_ChR_3s\2026_0507_095718_SpINB_ChR_IS46338_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_peristimulus_plot.png

=== 2026_0507_102707_SpINB_ChR_IS46338_ChR_4d_F_Fly2_Trial2_3000ms_thorax_sp1_ON ===
  onset_s=30.0000  offset_s=33.0000  duration_s=3.0000  n_pulses=968
  plot_t0=25.0000  plot_t1=38.0000  rows=